# 04 — DeBERTa baseline: test metrics + summary table

Loads checkpoints from `03_train_deberta_baseline.ipynb` and prints a summary table in the **same format** as the TF-IDF baseline.

**Output:** console table + `runs/deberta_ce_summary.csv`

In [ ]:
%pip install -q "transformers>=4.36" "datasets>=2.16" sentencepiece scikit-learn pandas

In [ ]:
import os
import sys

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
)

if os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/ECS111FinalProject"
else:
    PROJECT_DIR = os.environ.get("ECS111_PROJECT_DIR")
    if not PROJECT_DIR:
        cwd = os.getcwd()
        if os.path.isdir(os.path.join(cwd, "data", "splits")):
            PROJECT_DIR = cwd
        elif os.path.isdir(os.path.join(os.path.dirname(cwd), "data", "splits")):
            PROJECT_DIR = os.path.dirname(cwd)
        else:
            PROJECT_DIR = cwd

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))
from eval_summary import make_rotation_row, print_summary, save_summary_csv, score_binary

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 384
BATCH_SIZE = 32
VARIANT = "deberta_ce"
SUMMARY_TITLE = "--- DEBERTA + CE BASELINE SUMMARY ---"
CHECKPOINT_ROOT = os.path.join(PROJECT_DIR, "checkpoints", VARIANT)
OUT_CSV = os.path.join(PROJECT_DIR, "runs", f"{VARIANT}_summary.csv")

os.chdir(PROJECT_DIR)
print("PROJECT_DIR:", PROJECT_DIR)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)


def safe_str(x):
    if x is None:
        return ""
    s = str(x).strip()
    if s.lower() in ("nan", "none", ""):
        return ""
    return s


def prep_batch(batch):
    titles, texts, labels = batch["title"], batch["text"], batch["label"]
    pieces = []
    for t, te in zip(titles, texts):
        piece = (safe_str(t) + " " + safe_str(te)).strip()
        pieces.append(piece if piece else safe_str(te))
    return {"text": pieces, "labels": labels}


def tokenize_batch(batch):
    enc = tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)
    enc["labels"] = batch["labels"]
    return enc


def load_test_ds(rotation, split_name):
    path = os.path.join(PROJECT_DIR, "data", "splits", f"rotation_{rotation}", f"{split_name}.csv")
    raw = load_dataset("csv", data_files={"test": path})["test"]
    drop = raw.column_names
    ds = raw.map(prep_batch, batched=True, remove_columns=drop)
    return ds.map(tokenize_batch, batched=True, remove_columns=["text"])


def predict_split(model, ds):
    trainer = Trainer(model=model, processing_class=tokenizer, data_collator=collator)
    out = trainer.predict(ds)
    logits = out.predictions
    labels = out.label_ids
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()[:, 1]
    return labels, preds, probs

In [ ]:
rows = []
missing = []

for rotation in range(4):
    ckpt = os.path.join(CHECKPOINT_ROOT, f"rotation_{rotation}", "best_hf")
    if not os.path.isdir(ckpt):
        missing.append(rotation)
        print(f"skip rotation_{rotation}: no checkpoint at {ckpt}")
        continue

    print(f"Evaluating rotation_{rotation} from {ckpt}...")
    model = AutoModelForSequenceClassification.from_pretrained(ckpt)
    ds_indist = load_test_ds(rotation, "test_indist")
    ds_cross = load_test_ds(rotation, "test_crossgen")

    y_i, p_i, pr_i = predict_split(model, ds_indist)
    y_c, p_c, pr_c = predict_split(model, ds_cross)

    si = score_binary(y_i, p_i, pr_i)
    sc = score_binary(y_c, p_c, pr_c)
    rows.append(make_rotation_row(rotation, si["f1"], si["auc"], sc["f1"], sc["auc"]))

    print(f"\n>>> Summary after rotation_{rotation} ({len(rows)}/4 done)")
    df = print_summary(SUMMARY_TITLE, rows)
    save_summary_csv(df, OUT_CSV)
    print("Saved:", OUT_CSV)

if missing:
    print("Train missing rotations with 03_train_deberta_baseline.ipynb")
elif not rows:
    print("(no checkpoints found to evaluate)")